# 10. Train-Test Split

**Statistical Foundations for Data Science — Notebook 10 of 12**

Every $R^2$ in Notebook 9 was computed on the **same rows used to fit the model**. That is
like marking your own exam with the answer sheet open: the score is real, but it does not
measure what you want it to measure.

The question that matters is not *"how well does the model fit the data I have?"* but
*"how well will it do on data it has never seen?"* This notebook is about answering that
honestly.

### What you will learn

1. Why training error is a **biased** estimate of future performance
2. The **train / validation / test** split and what each part is for
3. `train_test_split`: sizes, shuffling, seeds, and **stratification**
4. **Data leakage** — the single most common way real projects fool themselves
5. **k-fold cross-validation**, and why it beats one split
6. Variants: stratified k-fold, leave-one-out, **group** splits, **time-series** splits
7. **Nested cross-validation** for honest hyperparameter tuning
8. How much test data you actually need

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (train_test_split, KFold, StratifiedKFold,
                                     GroupKFold, TimeSeriesSplit,
                                     cross_val_score, GridSearchCV)
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor

rng = np.random.default_rng(seed=7)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)

---
## 10.1 Why training error lies

A model has two jobs: capture the **signal** (the pattern that generalises) and ignore the
**noise** (the part specific to these rows). Nothing stops it from fitting the noise too —
and noise-fitting *always* improves the training score while *hurting* real performance.

The demonstration below is the whole notebook in one cell: fit polynomials of increasing
degree, and watch training error fall monotonically while test error turns around.

In [ ]:
# One true function, a small noisy sample from it
def true_f(t):
    return np.sin(1.4 * t) + 0.25 * t

n = 40
x_all = rng.uniform(0, 7, n)
y_all = true_f(x_all) + rng.normal(0, 0.35, n)

X_tr, X_te, y_tr, y_te = train_test_split(x_all.reshape(-1, 1), y_all,
                                          test_size=0.35, random_state=1)
print(f"Training rows: {len(y_tr)},  test rows: {len(y_te)}")

degrees = range(1, 16)
train_err, test_err = [], []
for d in degrees:
    m = make_pipeline(PolynomialFeatures(d), LinearRegression()).fit(X_tr, y_tr)
    train_err.append(np.sqrt(mean_squared_error(y_tr, m.predict(X_tr))))
    test_err.append(np.sqrt(mean_squared_error(y_te, m.predict(X_te))))

plt.plot(degrees, train_err, "o-", color="steelblue", label="training RMSE")
plt.plot(degrees, test_err, "o-", color="crimson", label="test RMSE")
best = degrees[int(np.argmin(test_err))]
plt.axvline(best, color="black", ls="--", label=f"best test degree = {best}")
plt.yscale("log"); plt.xlabel("polynomial degree (model complexity)")
plt.ylabel("RMSE (log scale)"); plt.title("Training error always falls. Test error does not.")
plt.legend(fontsize=8); plt.show()

print(f"{'degree':>7} {'train RMSE':>12} {'test RMSE':>12}")
for d, a, b_ in zip(degrees, train_err, test_err):
    mark = "  <- best on test" if d == best else ""
    print(f"{d:>7} {a:>12.4f} {b_:>12.4f}{mark}")

In [ ]:
# What the extremes look like
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
grid = np.linspace(0, 7, 400).reshape(-1, 1)
for ax, d, name in zip(axes, [1, best, 15], ["underfit", "about right", "overfit"]):
    m = make_pipeline(PolynomialFeatures(d), LinearRegression()).fit(X_tr, y_tr)
    ax.plot(grid, true_f(grid.ravel()), color="black", lw=1.6, ls="--", label="true function")
    ax.plot(grid, m.predict(grid), color="crimson", lw=2, label=f"degree {d}")
    ax.scatter(X_tr, y_tr, s=28, color="steelblue", label="train")
    ax.scatter(X_te, y_te, s=28, color="darkorange", marker="^", label="test")
    ax.set_ylim(-3, 4)
    ax.set_title(f"{name} (degree {d})\ntrain RMSE {np.sqrt(mean_squared_error(y_tr, m.predict(X_tr))):.3f}, "
                 f"test {np.sqrt(mean_squared_error(y_te, m.predict(X_te))):.3f}", fontsize=9)
    ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
# The extreme case: a model that memorises perfectly and generalises not at all
tree = DecisionTreeRegressor(random_state=0).fit(X_tr, y_tr)     # unlimited depth
print(f"Fully grown decision tree")
print(f"  training R^2 = {tree.score(X_tr, y_tr):.4f}   <- perfect memorisation")
print(f"  test     R^2 = {tree.score(X_te, y_te):.4f}")
print("\nA training score of 1.000 is a warning sign, not an achievement.")

---
## 10.2 Train, validation, test — three different jobs

| Split | Typical share | Used for | How often may you touch it? |
|---|---|---|---|
| **Training** | 60–80% | Fitting parameters (coefficients, tree splits, weights) | constantly |
| **Validation** | 10–20% | Choosing hyperparameters, comparing models, feature selection | many times |
| **Test** | 10–20% | One final, unbiased estimate of performance | **once** |

Why three and not two? Because **every decision you make using a dataset burns some of its
independence**. If you pick the polynomial degree by test RMSE, that test RMSE is now an
optimistic number — you selected for it. The validation set absorbs that selection, leaving
the test set clean.

> **The rule:** the test set is a bank vault, not a workbench. Open it when the work is
> finished, and report whatever it says.

In practice, cross-validation (section 10.5) usually replaces a single fixed validation set,
but the *test* set stays held out either way.

In [ ]:
# A 60/20/20 split, done with two calls
X = x_all.reshape(-1, 1)
X_temp, X_test, y_temp, y_test = train_test_split(X, y_all, test_size=0.20, random_state=0)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp,
                                                  test_size=0.25, random_state=0)  # 0.25*0.8 = 0.2

print(f"train {len(y_train)}  validation {len(y_val)}  test {len(y_test)}  "
      f"(total {len(y_all)})")

# Step 1: choose the degree on VALIDATION only
val_scores = {}
for d in range(1, 12):
    m = make_pipeline(PolynomialFeatures(d), LinearRegression()).fit(X_train, y_train)
    val_scores[d] = np.sqrt(mean_squared_error(y_val, m.predict(X_val)))
chosen = min(val_scores, key=val_scores.get)
print("\nValidation RMSE by degree:")
for d, s in val_scores.items():
    print(f"  degree {d:>2}: {s:.4f}{'   <- chosen' if d == chosen else ''}")

# Step 2: refit on train+validation, then report ONCE on test
final = make_pipeline(PolynomialFeatures(chosen), LinearRegression()).fit(X_temp, y_temp)
print(f"\nFinal model: degree {chosen}")
print(f"  validation RMSE (used for selection, optimistic) = {val_scores[chosen]:.4f}")
print(f"  TEST RMSE (the honest number)                    = "
      f"{np.sqrt(mean_squared_error(y_test, final.predict(X_test))):.4f}")

---
## 10.3 `train_test_split` in detail

```python
train_test_split(X, y,
                 test_size=0.2,      # fraction (or an integer count)
                 random_state=42,    # reproducibility
                 shuffle=True,       # almost always yes -- but NOT for time series
                 stratify=y)         # keep class proportions -- for classification
```

Things worth knowing:

- **`shuffle=True` is the default** and is usually right, because data often arrives sorted
  (by date, by class, by customer id). Sorted data plus no shuffle equals a broken split.
- **`random_state` is not a formality.** With small data, different seeds give visibly
  different scores. Report a cross-validated mean, not a lucky seed.
- **`stratify=y` for classification, always.** Otherwise a rare class can be absent from
  one side.

In [ ]:
# How much does the seed matter? Small data: a lot.
scores = []
for seed in range(200):
    Xa, Xb, ya, yb = train_test_split(X, y_all, test_size=0.3, random_state=seed)
    m = make_pipeline(PolynomialFeatures(4), LinearRegression()).fit(Xa, ya)
    scores.append(r2_score(yb, m.predict(Xb)))
scores = np.array(scores)

plt.hist(scores, bins=30, color="steelblue", edgecolor="white")
plt.axvline(scores.mean(), color="crimson", lw=2, label=f"mean = {scores.mean():.3f}")
plt.xlabel("test R^2"); plt.ylabel("frequency")
plt.title("Same model, same data, 200 different random splits")
plt.legend(); plt.show()

print(f"Best  split: R^2 = {scores.max():.4f}")
print(f"Worst split: R^2 = {scores.min():.4f}")
print(f"Mean +/- sd: {scores.mean():.4f} +/- {scores.std():.4f}")
print("\nQuoting the best seed is cherry-picking. Quoting one arbitrary seed is a coin flip.")
print("This is precisely the problem cross-validation solves.")

In [ ]:
# Stratification on an imbalanced classification target
m_cls = 400
Xc = rng.normal(size=(m_cls, 3))
yc = (rng.random(m_cls) < 0.08).astype(int)          # only about 8% positives
print(f"Overall positive rate: {yc.mean():.4f} ({yc.sum()} positives of {m_cls})\n")

plain, strat = [], []
for seed in range(300):
    _, _, _, t1 = train_test_split(Xc, yc, test_size=0.25, random_state=seed)
    _, _, _, t2 = train_test_split(Xc, yc, test_size=0.25, random_state=seed, stratify=yc)
    plain.append(t1.mean()); strat.append(t2.mean())

print(f"{'':<14}{'mean rate':>12}{'sd':>10}{'min':>10}{'max':>10}{'% with 0 positives':>22}")
for name, arr in [("plain", np.array(plain)), ("stratified", np.array(strat))]:
    print(f"{name:<14}{arr.mean():>12.4f}{arr.std():>10.4f}{arr.min():>10.4f}"
          f"{arr.max():>10.4f}{(arr == 0).mean():>22.2%}")
print("\nWithout stratification some test sets contain no positives at all -- recall and")
print("ROC-AUC are then undefined or meaningless. Always pass stratify=y.")

---
## 10.4 Data leakage

**Leakage** is when information from outside the training set sneaks into training. The
symptom is beautiful validation scores and a model that fails in production. Four common
forms:

| Leak | Example | Fix |
|---|---|---|
| **Preprocessing leak** | Scaling / imputing / encoding fitted on all the data before splitting | Fit transformers inside a `Pipeline`, on the training fold only |
| **Feature leak** | A feature that encodes the answer (`days_until_churn`, `final_invoice_paid`) | Ask "would I know this at prediction time?" |
| **Duplicate leak** | The same customer or near-duplicate row in train and test | Group-aware splitting; deduplicate first |
| **Temporal leak** | Training on the future to predict the past | `TimeSeriesSplit` |

Let's make the preprocessing leak concrete, because it is the one people commit without
noticing.

In [ ]:
# A situation where scaling before splitting really matters: p >> n
m_leak, p_leak = 60, 400
X_big = rng.normal(size=(m_leak, p_leak))
y_big = rng.normal(size=m_leak)                # NO relationship whatsoever

# WRONG: scale using statistics computed from every row, including the test rows
X_scaled_all = StandardScaler().fit_transform(X_big)
Xa, Xb, ya, yb = train_test_split(X_scaled_all, y_big, test_size=0.3, random_state=0)
leaky = Ridge(alpha=1.0).fit(Xa, ya)

# RIGHT: split first, and let a Pipeline fit the scaler on the training part only
Xa2, Xb2, ya2, yb2 = train_test_split(X_big, y_big, test_size=0.3, random_state=0)
clean = make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(Xa2, ya2)

print(f"Leaky pipeline  test R^2 = {leaky.score(Xb, yb):+.4f}")
print(f"Clean pipeline  test R^2 = {clean.score(Xb2, yb2):+.4f}")
print("\nBoth should be about 0 or negative -- there is no signal in this data.")
print("The point is the mechanism: the scaler saw the test rows, so the test rows were")
print("no longer unseen. With time-based features or target encoding the same mistake")
print("produces spectacular, entirely fake scores.")

In [ ]:
# The Pipeline is the fix, and it composes with cross-validation correctly.
# Inside cross_val_score, the scaler is refitted on each training fold.
pipe = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
cv_clean = cross_val_score(pipe, X_big, y_big, cv=5, scoring="r2")

# Contrast with scaling once, outside the CV loop
cv_leaky = cross_val_score(Ridge(alpha=1.0), X_scaled_all, y_big, cv=5, scoring="r2")

print(f"Pipeline inside CV (correct) : mean R^2 = {cv_clean.mean():+.4f}")
print(f"Scaled once outside CV (leak): mean R^2 = {cv_leaky.mean():+.4f}")
print("\nRule: every fitted transformation belongs INSIDE the pipeline.")
print("If you call .fit() on anything before splitting, stop and think.")

In [ ]:
# Feature leak: a column that quietly contains the answer
mm = 1_000
churn = (rng.random(mm) < 0.3).astype(int)
support_calls = rng.poisson(2 + 3*churn, mm)                    # legitimate signal
cancellation_form_started = churn * (rng.random(mm) < 0.9)      # LEAK: only churners do this

leak_df = pd.DataFrame({"support_calls": support_calls,
                        "cancellation_form_started": cancellation_form_started.astype(int),
                        "churn": churn})

for cols, label in [(["support_calls"], "legitimate features only"),
                    (["support_calls", "cancellation_form_started"], "with the leaky feature")]:
    Xl, Xt, yl, yt = train_test_split(leak_df[cols], leak_df.churn,
                                      test_size=0.3, random_state=0, stratify=leak_df.churn)
    acc = LogisticRegression().fit(Xl, yl).score(Xt, yt)
    print(f"  {label:<28} test accuracy = {acc:.4f}")

print("\n96% accuracy looks like a triumph and is worthless: by the time somebody has")
print("started the cancellation form, you do not need a model to tell you they are leaving.")
print("The test to apply to every feature: WOULD THIS VALUE BE AVAILABLE, AND MEAN THE")
print("SAME THING, AT THE MOMENT I NEED THE PREDICTION?")

---
## 10.5 k-fold cross-validation

A single split wastes data and gives a noisy estimate. **k-fold cross-validation** fixes
both:

1. Shuffle and divide the data into $k$ equal **folds**
2. For each fold: train on the other $k-1$ folds, evaluate on the held-out fold
3. Average the $k$ scores; the spread tells you how stable the estimate is

Every row is used for training $k-1$ times and for validation exactly once.

**Choosing $k$:** 5 or 10 are the standard choices. Larger $k$ means more training data per
fold (less bias) but more compute and more correlated folds (variance does not keep
falling). $k = n$ is **leave-one-out**.

In [ ]:
model = make_pipeline(PolynomialFeatures(4), LinearRegression())

kf = KFold(n_splits=5, shuffle=True, random_state=0)
fold_scores = cross_val_score(model, X, y_all, cv=kf, scoring="r2")

print("Fold-by-fold R^2:")
for i, s in enumerate(fold_scores, 1):
    print(f"  fold {i}: {s:.4f}")
print(f"\nmean = {fold_scores.mean():.4f}   sd = {fold_scores.std():.4f}")
print(f"Report as: R^2 = {fold_scores.mean():.3f} +/- {fold_scores.std():.3f} (5-fold CV)")
print(f"\nCompare with the 200 single splits from 10.3: mean {scores.mean():.3f}, "
      f"sd {scores.std():.3f}")
print("The CV mean is a far more stable estimate than any one split.")

In [ ]:
# A helper to visualise exactly which rows each split uses
from matplotlib.colors import ListedColormap
cmap_cv = ListedColormap(["steelblue", "darkorange"])

def plot_cv_indices(cv, X_, y_, groups=None, ax=None, title=""):
    ax = ax or plt.gca()
    n_ = len(X_)
    i = 0
    for i, (tr, te) in enumerate(cv.split(X_, y_, groups)):
        marks = np.full(n_, np.nan)
        marks[tr] = 0
        marks[te] = 1
        ax.scatter(range(n_), [i + 0.5] * n_, c=marks, marker="_", lw=9,
                   cmap=cmap_cv, vmin=-0.2, vmax=1.2)
    ax.set_yticks(np.arange(i + 1) + 0.5)
    ax.set_yticklabels([f"split {j+1}" for j in range(i + 1)])
    ax.set_xlabel("row index"); ax.set_title(title, fontsize=10); ax.invert_yaxis()
    return ax

X_demo = np.arange(50).reshape(-1, 1)
y_demo = np.r_[np.zeros(35), np.ones(15)]          # sorted, imbalanced

fig, ax = plt.subplots(1, 2, figsize=(13, 3.2))
plot_cv_indices(KFold(n_splits=5, shuffle=False), X_demo, y_demo, ax=ax[0],
                title="KFold(shuffle=False): contiguous blocks")
plot_cv_indices(KFold(n_splits=5, shuffle=True, random_state=0), X_demo, y_demo, ax=ax[1],
                title="KFold(shuffle=True): scattered")
plt.tight_layout(); plt.show()
print("Blue = training rows, orange = validation rows, one row of dots per split.")
print("With shuffle=False and sorted data, split 5 would be entirely the minority class.")

### Cross-validation variants

| Splitter | Use when |
|---|---|
| `KFold` | Regression, rows independent |
| `StratifiedKFold` | **Classification** — preserves class balance in every fold (this is what `cv=5` uses automatically for classifiers) |
| `LeaveOneOut` | Very small datasets; nearly unbiased but high variance and $n$ fits |
| `GroupKFold` | Repeated measurements per subject/customer/store — keeps a group entirely on one side |
| `TimeSeriesSplit` | Any temporal ordering — never train on the future |
| `RepeatedKFold` | Want a tighter estimate: repeat k-fold with different shuffles |

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(11, 8))
plot_cv_indices(StratifiedKFold(n_splits=5, shuffle=True, random_state=0),
                X_demo, y_demo, ax=ax[0],
                title="StratifiedKFold: each fold keeps the 70/30 class mix")

groups = np.repeat(np.arange(10), 5)              # 10 customers, 5 records each
plot_cv_indices(GroupKFold(n_splits=5), X_demo, y_demo, groups=groups, ax=ax[1],
                title="GroupKFold: a customer's 5 rows never straddle the split")

plot_cv_indices(TimeSeriesSplit(n_splits=5), X_demo, y_demo, ax=ax[2],
                title="TimeSeriesSplit: train always precedes validation, window grows")
plt.tight_layout(); plt.show()

In [ ]:
# Why GroupKFold matters: leaky duplicates inflate the score
n_cust, per_cust = 60, 5
cust = np.repeat(np.arange(n_cust), per_cust)
cust_effect = rng.normal(0, 3, n_cust)                       # each customer has a level
Xg = rng.normal(size=(n_cust * per_cust, 4))
yg = cust_effect[cust] + Xg[:, 0] * 0.4 + rng.normal(0, 0.5, n_cust * per_cust)

plain_cv = cross_val_score(DecisionTreeRegressor(random_state=0), Xg, yg,
                           cv=KFold(5, shuffle=True, random_state=0), scoring="r2")
group_cv = cross_val_score(DecisionTreeRegressor(random_state=0), Xg, yg,
                           cv=GroupKFold(5), groups=cust, scoring="r2")

print(f"Ordinary KFold : R^2 = {plain_cv.mean():+.4f}  (rows from the same customer")
print(f"                        appear on both sides -- optimistic)")
print(f"GroupKFold     : R^2 = {group_cv.mean():+.4f}  (honest: unseen customers)")
print("\nIf your production question is 'how will this do on a NEW customer?', only the")
print("second number answers it.")

In [ ]:
# Temporal leak: shuffling a time series lets the model interpolate the answer
T = 300
t = np.arange(T)
series = 20 + 0.05*t + 6*np.sin(2*np.pi*t/40) + np.cumsum(rng.normal(0, 0.4, T))
lag_X = np.column_stack([np.roll(series, k) for k in (1, 2, 3)])[3:]
lag_y = series[3:]

shuffled = cross_val_score(Ridge(), lag_X, lag_y, cv=KFold(5, shuffle=True, random_state=0),
                           scoring="r2")
ordered = cross_val_score(Ridge(), lag_X, lag_y, cv=TimeSeriesSplit(5), scoring="r2")

print(f"Shuffled KFold  : R^2 = {shuffled.mean():.4f}   <- knows the future")
print(f"TimeSeriesSplit : R^2 = {ordered.mean():.4f}   <- forecasts forward only")

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(t, series, color="steelblue", lw=1.2)
for i, (tr, te) in enumerate(TimeSeriesSplit(5).split(lag_X)):
    ax.axvspan(te[0] + 3, te[-1] + 3, alpha=0.12, color="crimson")
ax.set_title("TimeSeriesSplit validation windows (shaded) always lie after their training data")
ax.set_xlabel("time"); plt.tight_layout(); plt.show()

---
## 10.6 Cross-validation for model selection — and nested CV

`GridSearchCV` wraps the loop: for each hyperparameter setting, run k-fold CV, then refit
the best setting on all the data you gave it.

But note the trap. If you pick hyperparameters by CV score and then **report that same CV
score**, it is optimistic — you selected the maximum of many noisy numbers. Two honest
options:

1. **Hold out a test set** before tuning, tune with CV on the rest, report on the test set
2. **Nested cross-validation** — an outer loop for evaluation, an inner loop for tuning

In [ ]:
# GridSearchCV with an outer held-out test set
X_pool, X_final_test, y_pool, y_final_test = train_test_split(X, y_all, test_size=0.25,
                                                              random_state=3)

grid = GridSearchCV(
    make_pipeline(PolynomialFeatures(), Ridge()),
    param_grid={"polynomialfeatures__degree": range(1, 12),
                "ridge__alpha": [0.001, 0.01, 0.1, 1.0, 10.0]},
    cv=KFold(5, shuffle=True, random_state=0),
    scoring="neg_root_mean_squared_error",
).fit(X_pool, y_pool)

print(f"Best parameters : {grid.best_params_}")
print(f"Best inner-CV RMSE (optimistic) : {-grid.best_score_:.4f}")
print(f"Held-out TEST RMSE (honest)     : "
      f"{np.sqrt(mean_squared_error(y_final_test, grid.predict(X_final_test))):.4f}")

top = (pd.DataFrame(grid.cv_results_)[["param_polynomialfeatures__degree",
                                       "param_ridge__alpha", "mean_test_score", "std_test_score"]]
       .assign(rmse=lambda d: -d.mean_test_score)
       .nsmallest(5, "rmse")[["param_polynomialfeatures__degree", "param_ridge__alpha",
                              "rmse", "std_test_score"]])
print("\nTop 5 settings by inner CV:")
print(top.to_string(index=False))

In [ ]:
# Nested cross-validation: tuning happens inside every outer fold
inner = KFold(4, shuffle=True, random_state=1)
outer = KFold(5, shuffle=True, random_state=2)

search = GridSearchCV(make_pipeline(PolynomialFeatures(), Ridge()),
                      {"polynomialfeatures__degree": range(1, 10),
                       "ridge__alpha": [0.01, 0.1, 1.0, 10.0]},
                      cv=inner, scoring="neg_root_mean_squared_error")

nested = cross_val_score(search, X, y_all, cv=outer, scoring="neg_root_mean_squared_error")

# The naive (biased) version: one CV, tuned and reported on the same folds
naive = -GridSearchCV(make_pipeline(PolynomialFeatures(), Ridge()),
                      {"polynomialfeatures__degree": range(1, 10),
                       "ridge__alpha": [0.01, 0.1, 1.0, 10.0]},
                      cv=outer, scoring="neg_root_mean_squared_error"
                      ).fit(X, y_all).best_score_

print(f"Naive  (tune and report on the same folds): RMSE = {naive:.4f}")
print(f"Nested (tuning inside each outer fold)    : RMSE = {-nested.mean():.4f} "
      f"+/- {nested.std():.4f}")
print("\nThe nested estimate is worse, and it is the one to believe. The gap is the")
print("'winner's curse' -- the optimism you buy by choosing the best of many options.")

---
## 10.7 How much data should the test set get?

The test set has one job: estimate a number precisely enough to make a decision. From
Notebook 4, the standard error of an accuracy estimate is
$\sqrt{p(1-p)/n_{\text{test}}}$.

- $n_{\text{test}} = 100$ → margin of error about ±10 percentage points
- $n_{\text{test}} = 1{,}000$ → about ±3 points
- $n_{\text{test}} = 10{,}000$ → about ±1 point

So the split ratio should depend on the **absolute** size of the data, not on tradition:

| Dataset size | Sensible approach |
|---|---|
| < 1,000 rows | Cross-validation for everything; a fixed test set is too noisy to be useful |
| 1,000 – 100,000 | 80/20 or 70/15/15, plus CV inside the training part |
| > 1,000,000 | 98/1/1 is plenty — 10,000 test rows already pin accuracy to ±1% |

In [ ]:
from scipy import stats as sps

print("Margin of error on an accuracy estimate near 0.85 (95% confidence):")
for n_test in (50, 100, 500, 1_000, 5_000, 10_000, 100_000):
    se = np.sqrt(0.85 * 0.15 / n_test)
    print(f"  n_test = {n_test:>7,}  ->  +/-{1.96*se*100:>5.2f} percentage points")

print("\nA model reported at '87% accuracy' on 100 test rows could really be anywhere")
print("from 80% to 94%. Two such models cannot be meaningfully ranked.")
print("\nAnd the split is a trade-off: bigger test set = more reliable estimate but")
print("less data to train on, so the model you are measuring is worse.")

In [ ]:
# See the trade-off directly on a small dataset
sizes = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
means, sds = [], []
for ts in sizes:
    vals = []
    for seed in range(150):
        Xa, Xb, ya, yb = train_test_split(X, y_all, test_size=ts, random_state=seed)
        m = make_pipeline(PolynomialFeatures(4), LinearRegression()).fit(Xa, ya)
        vals.append(r2_score(yb, m.predict(Xb)))
    means.append(np.mean(vals)); sds.append(np.std(vals))

plt.errorbar(sizes, means, yerr=sds, marker="o", capsize=4, color="steelblue")
plt.xlabel("test_size"); plt.ylabel("test R^2 (mean +/- sd over 150 seeds)")
plt.title("Small test set = noisy estimate; large test set = worse model")
plt.show()

print(f"{'test_size':>10} {'mean R^2':>10} {'sd':>8} {'train rows':>12}")
for ts, mu, sd in zip(sizes, means, sds):
    print(f"{ts:>10.1f} {mu:>10.4f} {sd:>8.4f} {int(len(y_all)*(1-ts)):>12}")

---
## 10.8 A checklist for an honest evaluation

1. **Split first.** Before you plot, scale, impute, encode, or select features.
2. **Stratify** for classification; use **groups** for repeated measures; use
   **time-ordered** splits for temporal data.
3. **Put every fitted transformation in a `Pipeline`** so cross-validation refits it per
   fold.
4. **Audit every feature** for availability at prediction time.
5. **Tune with cross-validation**, not with the test set.
6. **Touch the test set once** and report what it says, including the uncertainty.
7. **Report mean ± sd across folds**, not a single lucky number.
8. If the test score is far worse than the CV score, suspect leakage or distribution shift —
   not bad luck.

---
## Exercises

**Exercise 1.** Using the bundled breast cancer dataset, compare a plain split with a
stratified split, and a single split with 10-fold cross-validation. Report which estimate
you would put in a report and why.

In [ ]:
# --- Solution 1 -------------------------------------------------------------
from sklearn.datasets import load_breast_cancer

bc = load_breast_cancer()
Xb_, yb_ = bc.data, bc.target
print(f"{Xb_.shape[0]} samples, {Xb_.shape[1]} features, positive rate {yb_.mean():.3f}\n")

clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))

single = []
for seed in range(50):
    Xa, Xt, ya, yt = train_test_split(Xb_, yb_, test_size=0.25, random_state=seed,
                                      stratify=yb_)
    single.append(clf.fit(Xa, ya).score(Xt, yt))
single = np.array(single)

cv10 = cross_val_score(clf, Xb_, yb_, cv=StratifiedKFold(10, shuffle=True, random_state=0))

print(f"Single 75/25 splits (50 seeds): {single.mean():.4f} +/- {single.std():.4f}  "
      f"(range {single.min():.4f}-{single.max():.4f})")
print(f"10-fold stratified CV         : {cv10.mean():.4f} +/- {cv10.std():.4f}")
print(f"\nA single split could have reported anything from {single.min():.1%} to "
      f"{single.max():.1%}.")
print("Report the cross-validated mean with its spread: it uses all the data for")
print("evaluation and is reproducible without depending on a seed.")

**Exercise 2.** Find the leak. The code below reports 100% accuracy on a customer-churn
model. Identify every problem and produce a corrected evaluation.

In [ ]:
# --- Solution 2: the broken version, for reference --------------------------
mm2 = 800
cust_id = np.repeat(np.arange(mm2 // 4), 4)                       # 4 records per customer
truth = (rng.random(mm2 // 4) < 0.35).astype(int)[cust_id]
data2 = pd.DataFrame({
    "customer_id": cust_id,
    "tenure": rng.uniform(1, 60, mm2),
    "monthly_spend": rng.normal(50, 12, mm2),
    "complaints": rng.poisson(1 + 2*truth, mm2),
    "exit_survey_completed": truth * (rng.random(mm2) < 0.85),    # LEAK
    "churn": truth,
})

# The broken pipeline: scale everything, shuffle-split, keep the leaky feature
bad_features = ["tenure", "monthly_spend", "complaints", "exit_survey_completed"]
X_all_scaled = StandardScaler().fit_transform(data2[bad_features])      # leak 1
Xa, Xt, ya, yt = train_test_split(X_all_scaled, data2.churn, test_size=0.25,
                                  random_state=0)                       # leak 2 (no groups)
print(f"BROKEN evaluation accuracy: "
      f"{LogisticRegression().fit(Xa, ya).score(Xt, yt):.4f}")

In [ ]:
# --- Solution 2: the three problems and the fix -----------------------------
print("Problems:")
print("  1. FEATURE LEAK   -- exit_survey_completed is only recorded for people who left")
print("  2. PREPROCESSING  -- StandardScaler was fitted on all rows before splitting")
print("  3. DUPLICATE LEAK -- the same customer's 4 records land on both sides of the split")
print()

good_features = ["tenure", "monthly_spend", "complaints"]
pipe2 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

# Fix 1+2: drop the leak, put the scaler in a pipeline
naive_cv = cross_val_score(pipe2, data2[good_features], data2.churn,
                           cv=StratifiedKFold(5, shuffle=True, random_state=0))
# Fix 3: group-aware splitting
group_cv = cross_val_score(pipe2, data2[good_features], data2.churn,
                           cv=GroupKFold(5), groups=data2.customer_id)

print(f"Drop the leaky feature + pipeline, ordinary CV : {naive_cv.mean():.4f} "
      f"+/- {naive_cv.std():.4f}")
print(f"Also group by customer (the honest number)    : {group_cv.mean():.4f} "
      f"+/- {group_cv.std():.4f}")
print("\nThe honest estimate answers the real question: how well do we predict churn for")
print("a customer we have never seen? That is the number the business should plan with.")

**Exercise 3.** You have 300 daily sales observations and want to forecast tomorrow.
(a) Show numerically why `KFold(shuffle=True)` gives a misleading score.
(b) Build a proper expanding-window evaluation with `TimeSeriesSplit` and report RMSE per
fold.
(c) Compare against the naive "tomorrow = today" baseline.

In [ ]:
# --- Solution 3 -------------------------------------------------------------
T3 = 300
trend = 0.08 * np.arange(T3)
seasonal = 5 * np.sin(2*np.pi*np.arange(T3)/7)
sales3 = 50 + trend + seasonal + np.cumsum(rng.normal(0, 0.6, T3))

LAGS = 7
X3 = np.column_stack([np.roll(sales3, k) for k in range(1, LAGS + 1)])[LAGS:]
y3 = sales3[LAGS:]

mdl = Ridge(alpha=1.0)
shuf = cross_val_score(mdl, X3, y3, cv=KFold(5, shuffle=True, random_state=0),
                       scoring="neg_root_mean_squared_error")
print(f"(a) Shuffled KFold RMSE  = {-shuf.mean():.4f}")
print("    Misleading because each validation point sits BETWEEN training points in")
print("    time -- the model interpolates rather than forecasts, which is not the task.\n")

tss = TimeSeriesSplit(n_splits=5)
print("(b) Expanding-window evaluation:")
rmses = []
for i, (tr, te) in enumerate(tss.split(X3), 1):
    m = Ridge(alpha=1.0).fit(X3[tr], y3[tr])
    rmse = np.sqrt(mean_squared_error(y3[te], m.predict(X3[te])))
    rmses.append(rmse)
    print(f"    fold {i}: train {len(tr):>3} rows, test {len(te):>3} rows, RMSE = {rmse:.4f}")
print(f"    mean RMSE = {np.mean(rmses):.4f} +/- {np.std(rmses):.4f}")

naive_rmse = []
for tr, te in tss.split(X3):
    naive_rmse.append(np.sqrt(mean_squared_error(y3[te], X3[te][:, 0])))   # lag-1 = yesterday
print(f"\n(c) Naive 'tomorrow = today' baseline RMSE = {np.mean(naive_rmse):.4f}")
better = np.mean(naive_rmse) - np.mean(rmses)
print(f"    The model beats the baseline by {better:.4f} RMSE "
      f"({better/np.mean(naive_rmse)*100:.1f}%).")
print("    Always report a naive baseline for time series -- persistence is hard to beat,")
print("    and a model that loses to it has no business in production.")

**Exercise 4 (challenge).** Quantify the "winner's curse" in model selection. Generate a
dataset with **no signal**, then select the best of $M$ candidate models by cross-validation
and record the winning CV score. Show that the winning score grows with $M$ even though no
model is better than chance, and that a held-out test set exposes it.

In [ ]:
# --- Solution 4 -------------------------------------------------------------
def winners_curse(M, n_rows=200, n_feat=5, reps=60):
    '''Best-of-M CV score vs its honest test score, on pure noise.'''
    best_cv, matched_test = [], []
    for _ in range(reps):
        Xw = rng.normal(size=(n_rows, n_feat))
        yw = rng.integers(0, 2, n_rows)                        # no relationship
        Xa, Xt, ya, yt = train_test_split(Xw, yw, test_size=0.4, random_state=0,
                                          stratify=yw)
        cvs, models = [], []
        for j in range(M):
            # M candidate models: different regularisation strengths and feature subsets
            cols = rng.choice(n_feat, size=rng.integers(1, n_feat + 1), replace=False)
            C = float(10 ** rng.uniform(-3, 2))
            mdl = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=500))
            s = cross_val_score(mdl, Xa[:, cols], ya,
                                cv=StratifiedKFold(5, shuffle=True, random_state=1)).mean()
            cvs.append(s); models.append((mdl, cols))
        j_best = int(np.argmax(cvs))
        best_cv.append(cvs[j_best])
        mdl, cols = models[j_best]
        matched_test.append(mdl.fit(Xa[:, cols], ya).score(Xt[:, cols], yt))
    return np.mean(best_cv), np.mean(matched_test)

print("Data contains NO signal, so honest accuracy should be about 0.50.")
print(f"{'candidates M':>13} {'best CV score':>15} {'its test score':>16}")
for M in (1, 3, 10, 30):
    cv_s, te_s = winners_curse(M)
    print(f"{M:>13} {cv_s:>15.4f} {te_s:>16.4f}")

print("\nThe best-of-M CV score climbs steadily above 0.50 as M grows -- that is pure")
print("selection optimism, not learning. The test column stays near chance and tells")
print("the truth.")
print("\nPractical implications:")
print("  * the more models you try, the more you must distrust the winning CV score")
print("  * keep a test set you never tune on, or use nested CV")
print("  * prefer the SIMPLEST model within one standard error of the best CV score")
print("    (the 'one standard error rule') rather than the raw maximum")

---
## Summary

| Concept | Key point |
|---|---|
| Training error | Always optimistic; falls with complexity forever |
| Train / validation / test | Fit / choose / report once |
| `train_test_split` | Shuffle by default; `stratify=y` for classification |
| Seed sensitivity | Small data → a single split is a coin flip |
| Leakage | Preprocessing, features, duplicates, time — the four classic forms |
| `Pipeline` | Puts every fitted transform inside the fold; the standard defence |
| k-fold CV | Every row validates once; report mean ± sd; $k=5$ or $10$ |
| `StratifiedKFold` | Default for classifiers |
| `GroupKFold` | Repeated measures — split by subject, not by row |
| `TimeSeriesSplit` | Never train on the future |
| Nested CV | Honest performance when you also tune hyperparameters |
| Test-set size | Choose it from the margin of error you need |

**Next up:** [Notebook 11 — Overfitting](11.%20Overfitting.ipynb). We now have an honest
measuring instrument; next we diagnose and cure the disease it detects.